# Introduction

Ce notebook présente un projet complet de **détection de fraude par carte bancaire** avec apprentissage supervisé, dans une logique **CRISP-DM** adaptée à un rendu académique.

**Objectif métier :** prédire les transactions frauduleuses (`Class = 1`) à partir des variables du jeu de données.

**Points méthodologiques clés :**
- Prétraitement et **séparation train / test** avant toute rééchantillonnage pour éviter la fuite d’information.
- **Pipeline** `imblearn` : `SimpleImputer` → `StandardScaler` → `SMOTE` → classifieur (SMOTE uniquement lors de l’apprentissage sur le train).
- **Optimisation** du **F1-score** (adapté au fort déséquilibre des classes) via `GridSearchCV` pour la forêt aléatoire.
- **Sauvegarde** des résultats (CSV, figures, modèle) pour reproduction et rapport.

**Fichier de données attendu :** `creditcard.csv` (à placer à la racine du projet ou adapter `DATA_PATH` ci-dessous).

# Data Understanding

Chargement du jeu de données, aperçu des colonnes, types et statistiques descriptives.

In [ ]:
from __future__ import annotations

import json
import warnings
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

from IPython.display import display

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings("ignore", category=RuntimeWarning)

RANDOM_STATE = 42
TARGET_COL = "Class"
DATA_PATH = Path("creditcard.csv")
OUTPUT_DIR = Path("output")
TEST_SIZE = 0.2
CV_FOLDS = 5

OUTPUT_DIR.mkdir(exist_ok=True)
plt.rcParams["figure.figsize"] = (10, 6)
sns.set_style("whitegrid")

np.random.seed(RANDOM_STATE)

In [ ]:
df = pd.read_csv(DATA_PATH)
df.columns = df.columns.str.strip()

if TARGET_COL not in df.columns:
    raise ValueError(f"Colonne cible « {TARGET_COL} » introuvable. Colonnes : {list(df.columns)}")

print("Dimensions :", df.shape)
display(df.head(10))

In [ ]:
df.info()

In [ ]:
display(df.describe().T)

## Exploratory Data Analysis (EDA)

Analyse de la qualité des données, de la **distribution de la cible** (fraude vs normal), des **corrélations** et de la **forme des variables** (notamment `Amount` et `Time`, souvent interprétables côté métier).

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(4)
missing_df = pd.DataFrame({"manquantes": missing, "%": missing_pct})
print("Valeurs manquantes par colonne (aperçu non nul) :")
display(missing_df[missing_df["manquantes"] > 0])
if missing_df["manquantes"].sum() == 0:
    print("Aucune valeur manquante détectée.")


In [ ]:
def plot_class_distribution(y: pd.Series, out_path: Path, title: str) -> None:
    fig, ax = plt.subplots(figsize=(8, 5))
    counts = y.value_counts().sort_index()
    labels = [str(i) for i in counts.index]
    colors = ["#4C72B0", "#DD8452"][: len(counts)]
    bars = ax.bar(labels, counts.values, color=colors)
    ax.set_xlabel("Class (0 = légitime, 1 = fraude)")
    ax.set_ylabel("Effectif")
    ax.set_title(title)
    for b, v in zip(bars, counts.values):
        ax.text(b.get_x() + b.get_width() / 2, v, f"{v:,}", ha="center", va="bottom", fontsize=10)
    plt.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.show()
    plt.close(fig)


y_full = df[TARGET_COL].astype(int)
plot_class_distribution(
    y_full,
    OUTPUT_DIR / "class_distribution.png",
    "Distribution de la variable cible (déséquilibre des classes)",
)
print("Taux de fraude :", f"{y_full.mean() * 100:.4f}%")

In [ ]:
def plot_correlation_heatmap(data: pd.DataFrame, out_path: Path) -> None:
    numeric = data.select_dtypes(include=[np.number])
    if numeric.shape[1] < 2:
        return
    cm = numeric.corr()
    n = cm.shape[0]
    figsize = (min(16, max(10, n * 0.35)), min(14, max(8, n * 0.35)))
    fig, ax = plt.subplots(figsize=figsize)
    sns.heatmap(
        cm,
        ax=ax,
        cmap="RdBu_r",
        center=0,
        square=True,
        linewidths=0.2,
        cbar_kws={"shrink": 0.6},
    )
    ax.set_title("Matrice de corrélation (variables numériques)")
    plt.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.show()
    plt.close(fig)


feat_for_corr = df.drop(columns=[TARGET_COL], errors="ignore")
plot_correlation_heatmap(feat_for_corr, OUTPUT_DIR / "correlation_heatmap.png")

In [ ]:
def plot_feature_distributions(df_in: pd.DataFrame, out_path: Path, max_cols: int = 12) -> None:
    num = df_in.select_dtypes(include=[np.number]).columns.tolist()
    if TARGET_COL in num:
        num.remove(TARGET_COL)
    priority = [c for c in ["Time", "Amount"] if c in num]
    rest = [c for c in num if c not in priority]
    cols = priority + rest[: max(0, max_cols - len(priority))]
    if not cols:
        return
    n = len(cols)
    ncols = 4
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(14, 3 * nrows))
    axes = np.atleast_1d(axes).ravel()
    for i, col in enumerate(cols):
        ax = axes[i]
        sns.histplot(df_in[col], kde=True, ax=ax, color="#4C72B0", edgecolor=None)
        ax.set_title(col, fontsize=9)
    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)
    fig.suptitle("Distributions de variables (échantillon)", y=1.02, fontsize=12)
    plt.tight_layout()
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)


plot_feature_distributions(df, OUTPUT_DIR / "feature_distributions.png")

# Data Preparation

- Gestion des valeurs manquantes via **`SimpleImputer` (médiane)** en première étape du pipeline : les statistiques d’imputation sont apprises **uniquement sur le train** (chaque pli de validation croisée).
- Les variables sont **toutes numériques** : pas d’encodage catégoriel nécessaire.
- **Normalisation** et **SMOTE** sont intégrés dans le pipeline d’apprentissage (section Modeling), après le **découpage 80/20 stratifié**.

In [ ]:
feature_cols = [c for c in df.columns if c != TARGET_COL]
X = df[feature_cols].copy()
y = df[TARGET_COL].astype(int)

n_missing = int(X.isnull().sum().sum())
print(f"Nombre total de valeurs manquantes (avant pipeline) : {n_missing}")
print("L’imputation médiane est appliquée dans le pipeline après la séparation train/test.")

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)
print("Train :", X_train.shape, "| Test :", X_test.shape)
print("Fraude (train) :", f"{y_train.mean() * 100:.4f}%")

## Gestion du déséquilibre — visualisation SMOTE (train uniquement)

Le **test** reste **non rééchantillonné** : il reflète la répartition réelle des transactions. Le graphique ci-dessous compare les effectifs **avant** et **après SMOTE** sur le **jeu d’entraînement** après **imputation médiane** puis **normalisation** (même ordre que dans le pipeline complet).

In [ ]:
_n_fraud = int(y_train.sum())
_kn = min(5, max(1, _n_fraud - 1))
imputer_vis = SimpleImputer(strategy="median")
X_train_imp = imputer_vis.fit_transform(X_train)
scaler_vis = StandardScaler()
X_train_scaled = scaler_vis.fit_transform(X_train_imp)
smote_vis = SMOTE(random_state=RANDOM_STATE, k_neighbors=_kn)
_, y_resampled = smote_vis.fit_resample(X_train_scaled, y_train)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, series, title in zip(
    axes,
    [y_train, pd.Series(y_resampled)],
    ["Avant SMOTE (train)", "Après SMOTE (train)"],
):
    c = series.value_counts().sort_index()
    ax.bar([str(i) for i in c.index], c.values, color=["#4C72B0", "#DD8452"][: len(c)])
    ax.set_title(title)
    ax.set_xlabel("Class")
    ax.set_ylabel("Effectif")
plt.suptitle("Distribution des classes — entraînement uniquement", y=1.05)
plt.tight_layout()
fig.savefig(OUTPUT_DIR / "smote_class_distribution.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close(fig)

# Modeling

Chaque modèle est un **Pipeline** `imblearn` :

`SimpleImputer` (médiane) → `StandardScaler` → `SMOTE` → **classifieur**

Ainsi, l’imputation, la normalisation et le sur-échantillonnage ne sont appris **que sur le train** (y compris dans les plis de validation croisée de `GridSearchCV`).

**Réglage d’hyperparamètres :** `GridSearchCV` avec métrique **F1** pour la **Random Forest** (minimum requis). Les autres modèles utilisent des hyperparamètres de base raisonnables ; la forêt aléatoire est la principale cible d’optimisation.

In [ ]:
def make_pipeline(classifier) -> ImbPipeline:
    k = int(y_train.sum())
    kn = min(5, max(1, k - 1))
    return ImbPipeline(
        [
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("smote", SMOTE(random_state=RANDOM_STATE, k_neighbors=kn)),
            ("clf", classifier),
        ]
    )


def train_with_optional_grid(
    pipe: ImbPipeline,
    param_grid: Optional[Dict[str, List[Any]]],
) -> Tuple[Any, Optional[Dict[str, Any]]]:
    cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    if param_grid:
        gs = GridSearchCV(
            pipe,
            param_grid,
            scoring="f1",
            cv=cv,
            n_jobs=-1,
            refit=True,
            verbose=0,
        )
        gs.fit(X_train, y_train)
        return gs.best_estimator_, gs.best_params_
    pipe.fit(X_train, y_train)
    return pipe, None


def evaluate_model(model, X_te, y_te) -> Dict[str, float]:
    y_pred = model.predict(X_te)
    proba = model.predict_proba(X_te)[:, 1]
    return {
        "accuracy": float(accuracy_score(y_te, y_pred)),
        "precision": float(precision_score(y_te, y_pred, zero_division=0)),
        "recall": float(recall_score(y_te, y_pred, zero_division=0)),
        "f1_score": float(f1_score(y_te, y_pred, zero_division=0)),
        "roc_auc": float(roc_auc_score(y_te, proba)),
    }


rf_param_grid = {
    "clf__n_estimators": [100, 200],
    "clf__max_depth": [8, 16, None],
    "clf__min_samples_leaf": [1, 2, 4],
    "clf__class_weight": [None, "balanced"],
}

configs: List[Dict[str, Any]] = [
    {
        "name": "Logistic Regression",
        "key": "logistic_regression",
        "pipe": make_pipeline(
            LogisticRegression(
                max_iter=5000,
                random_state=RANDOM_STATE,
                class_weight="balanced",
                solver="lbfgs",
            )
        ),
        "grid": None,
    },
    {
        "name": "Decision Tree",
        "key": "decision_tree",
        "pipe": make_pipeline(
            DecisionTreeClassifier(
                random_state=RANDOM_STATE,
                max_depth=12,
                min_samples_leaf=4,
                class_weight="balanced",
            )
        ),
        "grid": None,
    },
    {
        "name": "Random Forest",
        "key": "random_forest",
        "pipe": make_pipeline(RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)),
        "grid": rf_param_grid,
    },
    {
        "name": "Gradient Boosting",
        "key": "gradient_boosting",
        "pipe": make_pipeline(GradientBoostingClassifier(random_state=RANDOM_STATE)),
        "grid": None,
    },
]

try:
    from xgboost import XGBClassifier

    configs.append(
        {
            "name": "XGBoost",
            "key": "xgboost",
            "pipe": make_pipeline(
                XGBClassifier(
                    random_state=RANDOM_STATE,
                    eval_metric="logloss",
                    n_estimators=200,
                    max_depth=4,
                    learning_rate=0.08,
                    scale_pos_weight=1.0,
                    n_jobs=-1,
                )
            ),
            "grid": None,
        }
    )
except ImportError:
    print("XGBoost non installé — modèle ignoré (Gradient Boosting sklearn conservé).")

rows = []
fitted: Dict[str, Any] = {}
for cfg in configs:
    model, best_params = train_with_optional_grid(cfg["pipe"], cfg["grid"])
    metrics = evaluate_model(model, X_test, y_test)
    rows.append(
        {
            "model": cfg["name"],
            "key": cfg["key"],
            **metrics,
            "best_params": json.dumps(best_params, ensure_ascii=False) if best_params else "",
        }
    )
    fitted[cfg["key"]] = model

results = pd.DataFrame(rows)
results_display = results.drop(columns=["key"], errors="ignore")
results_display.to_csv(OUTPUT_DIR / "model_results.csv", index=False)
print("Résultats enregistrés :", OUTPUT_DIR / "model_results.csv")

# Evaluation

Métriques sur le **jeu de test** (non vu, non sur-échantillonné). Pour la fraude, le **rappel** (fraude détectée) et le **F1** sont particulièrement importants sous déséquilibre fort.

In [ ]:
results_sorted = results.sort_values(["f1_score", "roc_auc"], ascending=[False, False]).reset_index(
    drop=True
)
comparison = results_sorted[["model", "accuracy", "precision", "recall", "f1_score", "roc_auc"]]
display(comparison)

best_key = results_sorted.iloc[0]["key"]
best_name = results_sorted.iloc[0]["model"]
best_model = fitted[best_key]
print(f"\nMeilleur modèle (tri F1 puis ROC-AUC) : {best_name}")

joblib.dump(best_model, OUTPUT_DIR / "best_model.pkl")
print("Modèle sauvegardé :", OUTPUT_DIR / "best_model.pkl")

# Results

Visualisations pour le **meilleur modèle** : matrice de confusion, courbe ROC, importance des variables (coefficients ou importances selon l’estimateur).

In [ ]:
feature_names = feature_cols


def plot_confusion_matrix_fig(y_true, y_pred, out_path: Path, title: str) -> None:
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        ax=ax,
        xticklabels=["Prédit 0", "Prédit 1"],
        yticklabels=["Réel 0", "Réel 1"],
    )
    ax.set_title(title)
    ax.set_ylabel("Vérité terrain")
    ax.set_xlabel("Prédiction")
    plt.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.show()
    plt.close(fig)


def plot_roc_fig(y_true, y_score, out_path: Path, title: str) -> None:
    fpr, tpr, _ = roc_curve(y_true, y_score)
    auc = roc_auc_score(y_true, y_score)
    fig, ax = plt.subplots(figsize=(7, 6))
    ax.plot(fpr, tpr, label=f"ROC (AUC = {auc:.4f})")
    ax.plot([0, 1], [0, 1], "k--", label="Hasard")
    ax.set_xlabel("Taux de faux positifs")
    ax.set_ylabel("Taux de vrais positifs")
    ax.set_title(title)
    ax.legend(loc="lower right")
    plt.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.show()
    plt.close(fig)


def extract_importances(model, names: List[str]) -> np.ndarray:
    clf = model.named_steps["clf"]
    if hasattr(clf, "feature_importances_"):
        return np.asarray(clf.feature_importances_)
    if hasattr(clf, "coef_"):
        return np.asarray(np.abs(clf.coef_).ravel())
    return np.zeros(len(names))


def plot_importance_fig(importances: np.ndarray, names: List[str], out_path: Path, top_n: int = 20) -> None:
    order = np.argsort(importances)[::-1][:top_n]
    fig, ax = plt.subplots(figsize=(10, 7))
    y_pos = np.arange(len(order))
    ax.barh(y_pos, importances[order], color="#4C72B0")
    ax.set_yticks(y_pos)
    ax.set_yticklabels([names[i] for i in order], fontsize=9)
    ax.invert_yaxis()
    ax.set_xlabel("Importance (|coefficient| ou importance arbre)")
    ax.set_title(f"Top {top_n} variables — {best_name}")
    plt.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.show()
    plt.close(fig)


y_pred_best = best_model.predict(X_test)
y_proba_best = best_model.predict_proba(X_test)[:, 1]

plot_confusion_matrix_fig(
    y_test,
    y_pred_best,
    OUTPUT_DIR / "confusion_matrix.png",
    f"Matrice de confusion — {best_name}",
)
plot_roc_fig(
    y_test,
    y_proba_best,
    OUTPUT_DIR / "roc_curve.png",
    f"Courbe ROC — {best_name}",
)

imp = extract_importances(best_model, feature_names)
plot_importance_fig(imp, feature_names, OUTPUT_DIR / "feature_importance.png")

## Interprétation métier

### Pourquoi le déséquilibre des classes compte

Les transactions frauduleuses sont **très minoritaires**. Un modèle naïf qui prédit toujours « non fraude » peut afficher une **accuracy élevée** tout en **ratant toutes les fraudes**. Le déséquilibre doit donc être traité (ici **SMOTE** sur le train, **poids de classe** ou **scale_pos_weight** selon les modèles) et les performances doivent être lues avec des métriques adaptées.

### Pourquoi le F1-score est important

Le **F1** est la moyenne harmonique de la **précision** et du **rappel**. Il sanctionne à la fois les modèles qui déclenchent trop de fausses alertes (précision faible) et ceux qui **manquent des fraudes** (rappel faible). C’est un compromis pertinent quand les deux erreurs ont un coût métier.

### Impact métier des faux négatifs (fraude non détectée)

Un **faux négatif** correspond à une **fraude réelle classée comme transaction légitime** : pertes financières directes, charge sur la banque ou le commerçant, dégradation de la confiance et risque réglementaire. En pratique, on cherche souvent un **rappel élevé** sur la fraude, quitte à accepter davantage de **faux positifs** (transactions bloquées à tort), selon la politique de risque.

### Pourquoi le modèle retenu est le meilleur candidat (dans ce notebook)

Le modèle est choisi par **tri décroissant du F1-score sur le test**, puis **ROC-AUC** en cas d’égalité, après **GridSearchCV** sur la **Random Forest** (optimisation du F1 en validation croisée). Cette procédure favorise un équilibre entre **détection des fraudes** et **maîtrise des fausses alertes**, tout en vérifiant la **discrimination globale** (AUC) sur des données non rééchantillonnées.

# Conclusion

- Pipeline **StandardScaler → SMOTE → classifieur** avec entraînement sur le **train** seulement ; évaluation sur un **test** stratifié **80/20**.
- **Random Forest** soumise à **GridSearchCV** (métrique **F1**), en complément de régressions logistiques, arbre, gradient boosting et XGBoost si disponible.
- Fichiers produits dans `output/` : `model_results.csv`, `best_model.pkl`, `class_distribution.png`, `correlation_heatmap.png`, `smote_class_distribution.png`, `feature_distributions.png`, `confusion_matrix.png`, `roc_curve.png`, `feature_importance.png`.

**Limites usuelles :** le jeu `creditcard` est déjà transformé (PCA sur une partie des variables) — l’interprétation « métier » des `V*` est limitée sans retrouver les variables d’origine. Les performances sur le test dépendent aussi de la **dérive temporelle** (`Time`) en production.

---

*Projet prêt pour soumission académique — exécuter toutes les cellules dans l’ordre après avoir placé `creditcard.csv` à la racine du projet.*